# e6 – Overall comparison to the benchmark (`hist_rw`)

Does each model forecast next month's realised variance better than the historical-volatility
benchmark `hist_rw` (the forecast for month t+1 is month t's realised variance)?
Models come from `research_config.SELECTED`; each is compared with `BENCHMARK`.

**Tests:**
Every Elastic Net variant includes `log_rv_1m`, so it nests `hist_rw` (intercept 0, coefficient 1
on `log_rv_1m`, 0 on the rest). For these, log-RV MSE uses Clark-West, with DM for the raw loss
difference. GARCH, GARCH-MIDAS and XGBoost do not nest `hist_rw`, so log-RV MSE uses DM only.
QLIKE uses DM for every model. Pooled differentials are averaged across ETFs each month and tested
with Newey-West (Andrews bandwidth). P-values are one-sided, H1: `a` beats `b`.

Clark-West is approximate for tuned Elastic Net and can be significant despite
worse raw MSE. DM p-values also require caution for nested comparisons.

Output in `forecasting/results/`: `e6_losses.csv`, `e6_evaluation.csv` 
`a` is the model and `b` is `hist_rw`. `gain_t > 0` favours `a`; `loss_ratio` is loss(a) / loss(b).
`pct_change = (loss_ratio - 1) * 100`; negative means lower raw loss for `a`.

In [9]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import sys
from importlib import reload
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

from utils.paths import FORECAST_DIR, FORECAST_RESULTS_DIR as RESULTS_DIR
from utils.portfolio_utils import FORECAST_FILES
import research_config

# Notebook kernels cache earlier comparison definitions between runs.
reload(research_config)
from research_config import SELECTED, BENCHMARK, TICKERS

RESULTS_DIR.mkdir(exist_ok=True)

# Families whose variants nest hist_rw: each Elastic Net includes log_rv_1m.
NESTED_FAMILIES = {"Elastic Net"}

frames = []
for fam in SELECTED:
    f = pd.read_csv(FORECAST_DIR / FORECAST_FILES[fam], parse_dates=["date"]).assign(family=fam)
    # Older exports may omit target_date; all forecasts target the next month end.
    next_month = f["date"] + pd.offsets.MonthEnd(1)
    if "target_date" in f and not pd.to_datetime(f["target_date"]).eq(next_month).all():
        raise ValueError(f"{FORECAST_FILES[fam]} targets a month other than the next one.")
    frames.append(f.assign(target_date=next_month))
df = pd.concat(frames, ignore_index=True)

selected = [m for ms in SELECTED.values() for m in ms]
available = set(df["model"])
for m in selected:
    if m not in available:
        print(f"NOTICE: {m} has no forecasts yet; it and its pair are skipped")
models = [m for m in selected if m in available]
if BENCHMARK not in models:
    raise ValueError(f"{BENCHMARK} has no forecasts.")
df = df[df["model"].isin(models) & df["etf"].isin(TICKERS)].copy()
if df.duplicated(["model", "etf", "date"]).any():
    raise ValueError("Duplicate model/ETF/forecast-month rows.")
if df["date"].isna().any():
    raise ValueError("Forecast dates must be present.")

family = {m: fam for fam, ms in SELECTED.items() for m in ms}
nested = {m: family[m] in NESTED_FAMILIES for m in models}
pairs = [(f"E6.{i}", m, BENCHMARK, family[m])
         for i, m in enumerate([m for m in models if m != BENCHMARK], 1)]
print(f"{len(models)} of {len(selected)} models available; {len(pairs)} comparisons with {BENCHMARK}")

11 of 11 models available; 10 comparisons with hist_rw


In [10]:
def andrews_bandwidth(x):
    x = np.asarray(x, float)
    n = len(x)
    xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float)
    n = len(x)
    xd = x - x.mean()
    lag = max(0, min(lag, n - 1))
    s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where all models have matching target dates and realized values.
`XGBoost` starts one month earlier (2008-01), which is dropped.

In [11]:
d = df[df.groupby(["etf", "date"])["model"].transform("nunique") == len(models)].copy()
d = d.sort_values(["date", "etf", "model"]).reset_index(drop=True)
if d.empty:
    raise ValueError("No common forecast sample across the models.")
values = ["actual_logRV", "actual_RV", "fc_log", "fc_level"]
if not np.isfinite(d[values].to_numpy()).all():
    raise ValueError("The common sample contains non-finite forecasts or outcomes.")
if (d[["actual_RV", "fc_level"]] <= 0).any().any():
    raise ValueError("QLIKE requires positive realized and forecast variances.")
keys = ["etf", "date"]
if (d.groupby(keys)["target_date"].nunique() != 1).any():
    raise ValueError("Target months differ across models.")
for col in ["actual_logRV", "actual_RV"]:
    wide = d.pivot(index=keys, columns="model", values=col)
    if not np.allclose(wide.to_numpy(), wide[[BENCHMARK]].to_numpy(), rtol=1e-10, atol=1e-12):
        raise ValueError(f"Realized outcomes differ across models: {col}")
# Keep the pooled monthly average equally weighted across the five ETFs.
if (d.groupby("date")["etf"].nunique() != len(TICKERS)).any():
    raise ValueError("The common sample must contain all five ETFs each month.")

d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"Forecast origins: {d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}")
print(f"Target months: {d['target_date'].min():%Y-%m} to {d['target_date'].max():%Y-%m}")
print(f"{d['date'].nunique()} months × {len(TICKERS)} ETFs × {len(models)} models = {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e6_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model").loc[models, LOSSES].round(4)

Forecast origins: 2008-02 to 2025-11
Target months: 2008-03 to 2025-12
214 months × 5 ETFs × 11 models = 11770 rows


,QLIKE,MSE_log,MSE_level
model,,,
hist_rw,0.4063,0.6030,0.0001
garch,0.2635,0.4736,0.0001
garch_midas_baa10y,0.2682,0.4826,0.0001
en_no_macro,0.3266,0.5228,0.0001
en_no_vix,0.3797,0.5371,0.0004
en_vix_only,0.2913,0.4769,0.0015
en_full_macro,0.3247,0.4902,0.0011
xgb_no_macro,0.4104,0.5893,0.0001
xgb_no_vix,0.4416,0.5767,0.0001


## Tests

In [12]:
loss_idx = losses.set_index(["model", "ETF"])
rows = []
cw_components = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"]).sort_index()
    B = d[d["model"] == b].set_index(["date", "etf"]).sort_index()
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    mse_diff = A["MSE_log"] - B["MSE_log"]
    comparisons = [("QLIKE", "DM", dm_diff), ("MSE_log", "DM", mse_diff)]
    if nested[a]:
        # Clark-West adjusted differential; positive favours a.
        adjustment = (B["fc_log"] - A["fc_log"]) ** 2
        cw = -mse_diff + adjustment
        comparisons.append(("MSE_log", "Clark-West", cw))
        # Reuse these components in the diagnostic table.
        cw_components.append({
            "pair": pair_id, "a": a,
            "mean_raw_gain": (-mse_diff).groupby("date").mean().mean(),
            "mean_adjustment_term": adjustment.groupby("date").mean().mean(),
            "mean_CW_differential": cw.groupby("date").mean().mean(),
        })

    for etf in ["pooled"] + TICKERS:
        for loss, test, diff in comparisons:
            series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
            series = series.sort_index()
            if len(series) < 2:
                raise ValueError(f"Insufficient observations for {pair_id}, {etf}.")
            t = -hac_t(series) if test == "DM" else hac_t(series)
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": float(norm.sf(t)),
            })

tests = pd.DataFrame(rows)
tests["pct_change"] = (tests["loss_ratio"] - 1) * 100  # negative: a has the lower raw loss
tests.to_csv(RESULTS_DIR / "e6_evaluation.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True)

,pair,a,b,loss,test,loss_ratio,gain_t,p_value,pct_change
0,E6.1,garch,hist_rw,QLIKE,DM,0.6487,4.1034,0.0000,-35.1327
1,E6.1,garch,hist_rw,MSE_log,DM,0.7854,3.8433,0.0001,-21.4617
2,E6.2,garch_midas_baa10y,hist_rw,QLIKE,DM,0.6602,4.0567,0.0000,-33.9797
3,E6.2,garch_midas_baa10y,hist_rw,MSE_log,DM,0.8004,3.5340,0.0002,-19.9610
4,E6.3,en_no_macro,hist_rw,QLIKE,DM,0.8039,3.1976,0.0007,-19.6051
5,E6.3,en_no_macro,hist_rw,MSE_log,DM,0.8670,3.9078,0.0000,-13.3029
6,E6.3,en_no_macro,hist_rw,MSE_log,Clark-West,0.8670,6.0673,0.0000,-13.3029
7,E6.4,en_no_vix,hist_rw,QLIKE,DM,0.9346,0.5235,0.3003,-6.5389
8,E6.4,en_no_vix,hist_rw,MSE_log,DM,0.8907,1.7042,0.0442,-10.9272
9,E6.4,en_no_vix,hist_rw,MSE_log,Clark-West,0.8907,5.5433,0.0000,-10.9272


Clark-West components for the nested (Elastic Net) pairs, pooled across ETFs: the raw log-MSE gain
of `a` over `hist_rw`, the nesting adjustment, and their sum.

In [13]:
pd.DataFrame(cw_components).set_index("pair").round(4)

,a,mean_raw_gain,mean_adjustment_term,mean_CW_differential
pair,,,,
E6.3,en_no_macro,0.0802,0.0559,0.1361
E6.4,en_no_vix,0.0659,0.1431,0.2090
E6.5,en_vix_only,0.1261,0.1425,0.2686
E6.6,en_full_macro,0.1128,0.1839,0.2967


## By sector tests (exploratory)

Each cell is `loss_ratio (p-value)`; a ratio below 1 means the model is better than `hist_rw`.
Five sectors times ten models with no multiple-testing correction, so a single sector's p-value
is weak evidence; the pooled column is the formal result.

In [14]:
from IPython.display import display

SECTORS = {
    "XLB": "Materials", "XLE": "Energy", "XLF": "Financials",
    "XLI": "Industrials", "XLY": "Cons. Disc.",
}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss, test):
    t = tests[tests["loss"].eq(loss) & tests["test"].eq(test)].copy()
    t["cell"] = [f"{ratio:.3f} ({fmt_p(p)})" for ratio, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell").reindex(columns=TICKERS + ["pooled"])
    table = table.loc[t["row"].drop_duplicates()]
    table.columns = [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


for loss, test in [("MSE_log", "DM"), ("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sector(loss, test))

MSE_log: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch vs hist_rw,0.804 (0.001),0.826 (<0.001),0.762 (<0.001),0.726 (<0.001),0.827 (0.005),0.785 (<0.001)
E6.2: garch_midas_baa10y vs hist_rw,0.815 (0.003),0.848 (0.006),0.767 (<0.001),0.751 (<0.001),0.842 (0.008),0.800 (<0.001)
E6.3: en_no_macro vs hist_rw,0.854 (<0.001),0.884 (0.006),0.880 (<0.001),0.840 (<0.001),0.877 (<0.001),0.867 (<0.001)
E6.4: en_no_vix vs hist_rw,0.886 (0.065),0.941 (0.291),0.886 (0.041),0.839 (0.010),0.913 (0.076),0.891 (0.044)
E6.5: en_vix_only vs hist_rw,0.750 (<0.001),0.865 (0.009),0.757 (<0.001),0.719 (0.004),0.886 (0.095),0.791 (<0.001)
E6.6: en_full_macro vs hist_rw,0.777 (0.004),0.917 (0.220),0.812 (0.006),0.723 (0.002),0.854 (0.039),0.813 (0.008)
E6.7: xgb_no_macro vs hist_rw,0.977 (0.409),0.993 (0.477),0.942 (0.261),0.885 (0.055),1.111 (0.719),0.977 (0.395)
E6.8: xgb_no_vix vs hist_rw,1.035 (0.602),0.963 (0.379),0.962 (0.394),0.864 (0.059),0.970 (0.370),0.956 (0.335)
E6.9: xgb_vix_only vs hist_rw,0.914 (0.252),0.936 (0.304),0.830 (0.030),0.733 (<0.001),0.914 (0.140),0.859 (0.028)
E6.10: xgb_full_macro vs hist_rw,0.944 (0.341),0.935 (0.290),0.817 (0.029),0.783 (0.011),0.900 (0.149),0.868 (0.069)


MSE_log: ratio (p-value), Clark-West


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.3: en_no_macro vs hist_rw,0.854 (<0.001),0.884 (<0.001),0.880 (<0.001),0.840 (<0.001),0.877 (<0.001),0.867 (<0.001)
E6.4: en_no_vix vs hist_rw,0.886 (<0.001),0.941 (<0.001),0.886 (<0.001),0.839 (<0.001),0.913 (<0.001),0.891 (<0.001)
E6.5: en_vix_only vs hist_rw,0.750 (<0.001),0.865 (<0.001),0.757 (<0.001),0.719 (<0.001),0.886 (<0.001),0.791 (<0.001)
E6.6: en_full_macro vs hist_rw,0.777 (<0.001),0.917 (<0.001),0.812 (<0.001),0.723 (<0.001),0.854 (<0.001),0.813 (<0.001)


QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch vs hist_rw,0.672 (<0.001),0.733 (<0.001),0.581 (<0.001),0.621 (<0.001),0.674 (<0.001),0.649 (<0.001)
E6.2: garch_midas_baa10y vs hist_rw,0.664 (<0.001),0.759 (0.003),0.592 (<0.001),0.637 (<0.001),0.686 (<0.001),0.660 (<0.001)
E6.3: en_no_macro vs hist_rw,0.825 (0.009),0.851 (0.025),0.767 (0.003),0.807 (0.002),0.788 (<0.001),0.804 (<0.001)
E6.4: en_no_vix vs hist_rw,0.920 (0.277),1.062 (0.618),0.875 (0.185),0.940 (0.350),0.908 (0.169),0.935 (0.300)
E6.5: en_vix_only vs hist_rw,0.698 (<0.001),0.840 (0.033),0.666 (0.004),0.664 (<0.001),0.752 (0.003),0.717 (<0.001)
E6.6: en_full_macro vs hist_rw,0.779 (0.033),1.027 (0.551),0.746 (0.012),0.715 (<0.001),0.777 (0.003),0.799 (0.022)
E6.7: xgb_no_macro vs hist_rw,1.053 (0.624),1.114 (0.699),0.955 (0.402),0.839 (0.037),1.141 (0.721),1.010 (0.529)
E6.8: xgb_no_vix vs hist_rw,1.175 (0.759),1.164 (0.731),1.100 (0.686),0.978 (0.446),1.035 (0.605),1.087 (0.682)
E6.9: xgb_vix_only vs hist_rw,1.018 (0.532),1.075 (0.624),0.802 (0.058),0.749 (0.001),0.825 (0.059),0.880 (0.144)
E6.10: xgb_full_macro vs hist_rw,1.092 (0.645),1.102 (0.658),0.864 (0.163),0.920 (0.334),0.910 (0.275),0.965 (0.415)


## By sector percentage changes

Percentage change in mean loss for each model relative to `hist_rw`:
`100 × (loss_ratio - 1)`. Negative values mean lower loss than the benchmark;
positive values mean higher loss. Log MSE and QLIKE use the same common sample
as the tests above. The pooled column includes all five sectors.

In [15]:
for loss in ["MSE_log", "QLIKE"]:
    pct = (
        tests.loc[tests["loss"].eq(loss) & tests["test"].eq("DM")]
        .assign(row=lambda t: t["pair"] + ": " + t["a"])
        .pivot(index="row", columns="ETF", values="pct_change")
        .reindex(columns=TICKERS + ["pooled"])
    )
    pct = pct.loc[[f"{pair_id}: {a}" for pair_id, a, _, _ in pairs]]
    pct.columns = [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"]
    print(f"{'Log MSE' if loss == 'MSE_log' else loss}: % change relative to {BENCHMARK}")
    display(pct.rename_axis(None).map(lambda value: f"{value:+.2f}%"))

Log MSE: % change relative to hist_rw


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch,-19.63%,-17.36%,-23.77%,-27.37%,-17.33%,-21.46%
E6.2: garch_midas_baa10y,-18.47%,-15.24%,-23.31%,-24.87%,-15.78%,-19.96%
E6.3: en_no_macro,-14.57%,-11.59%,-12.04%,-16.03%,-12.31%,-13.30%
E6.4: en_no_vix,-11.44%,-5.88%,-11.38%,-16.10%,-8.68%,-10.93%
E6.5: en_vix_only,-25.03%,-13.45%,-24.26%,-28.13%,-11.39%,-20.91%
E6.6: en_full_macro,-22.29%,-8.26%,-18.83%,-27.74%,-14.59%,-18.71%
E6.7: xgb_no_macro,-2.34%,-0.69%,-5.76%,-11.50%,+11.06%,-2.27%
E6.8: xgb_no_vix,+3.49%,-3.65%,-3.79%,-13.58%,-3.04%,-4.36%
E6.9: xgb_vix_only,-8.65%,-6.36%,-16.98%,-26.70%,-8.63%,-14.12%
E6.10: xgb_full_macro,-5.64%,-6.54%,-18.35%,-21.71%,-10.00%,-13.17%


QLIKE: % change relative to hist_rw


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch,-32.78%,-26.67%,-41.89%,-37.92%,-32.64%,-35.13%
E6.2: garch_midas_baa10y,-33.62%,-24.14%,-40.76%,-36.27%,-31.38%,-33.98%
E6.3: en_no_macro,-17.50%,-14.85%,-23.25%,-19.32%,-21.22%,-19.61%
E6.4: en_no_vix,-8.05%,+6.22%,-12.48%,-5.96%,-9.23%,-6.54%
E6.5: en_vix_only,-30.21%,-15.99%,-33.37%,-33.59%,-24.83%,-28.29%
E6.6: en_full_macro,-22.14%,+2.67%,-25.38%,-28.52%,-22.31%,-20.08%
E6.7: xgb_no_macro,+5.32%,+11.37%,-4.53%,-16.10%,+14.13%,+1.01%
E6.8: xgb_no_vix,+17.53%,+16.40%,+9.95%,-2.19%,+3.53%,+8.69%
E6.9: xgb_vix_only,+1.78%,+7.54%,-19.78%,-25.13%,-17.55%,-11.98%
E6.10: xgb_full_macro,+9.19%,+10.23%,-13.57%,-8.00%,-9.02%,-3.47%


## Diagnostics: largest level errors

For each model, the share of its level MSE that comes from its five largest ETF-month errors,
then the ten largest level errors across all models.

In [16]:
level_errors = d.copy()
level_errors["level_ratio"] = level_errors.fc_level / level_errors.actual_RV
level_errors["level_error_share"] = (
    level_errors.MSE_level / level_errors.groupby("model").MSE_level.transform("sum")
)
top5_share = (level_errors.sort_values("MSE_level", ascending=False)
              .groupby("model").head(5).groupby("model")["level_error_share"].sum())
print("Share of level MSE from each model's five largest errors:")
display(top5_share.loc[models].round(3).to_frame())

error_cols = ["model", "etf", "date", "target_date", "actual_RV", "fc_log", "sigma2", "fc_level",
              "level_ratio", "MSE_log", "MSE_level", "QLIKE", "level_error_share"]
level_errors.nlargest(10, "MSE_level")[error_cols]

Share of level MSE from each model's five largest errors:


,level_error_share
model,
hist_rw,0.435
garch,0.504
garch_midas_baa10y,0.493
en_no_macro,0.523
en_no_vix,0.683
en_vix_only,0.922
en_full_macro,0.815
xgb_no_macro,0.491
xgb_no_vix,0.490


,model,etf,date,target_date,actual_RV,fc_log,sigma2,fc_level,level_ratio,MSE_log,MSE_level,QLIKE,level_error_share
476,en_vix_only,XLI,2008-10-31,2008-11-30,0.024007,-0.296203,0.247296,0.841512,35.053043,11.786961,0.668315,2.585391,0.426090
487,en_vix_only,XLY,2008-10-31,2008-11-30,0.045574,-0.392577,0.288833,0.780235,17.120142,7.267548,0.539726,1.898666,0.344108
484,en_full_macro,XLY,2008-10-31,2008-11-30,0.045574,-0.528812,0.329611,0.694887,15.247411,6.551574,0.421607,1.789995,0.366374
462,en_full_macro,XLF,2008-10-31,2008-11-30,0.087358,-0.929827,0.714462,0.564059,6.456855,2.273796,0.227244,1.020017,0.197474
739,en_no_vix,XLF,2009-03-31,2009-04-30,0.062968,-1.187178,0.748811,0.443626,7.045289,2.489937,0.144901,1.094298,0.302122
473,en_full_macro,XLI,2008-10-31,2008-11-30,0.024007,-1.069012,0.314917,0.401899,16.741049,7.077757,0.142803,1.877597,0.124095
443,en_vix_only,XLB,2008-10-31,2008-11-30,0.035719,-1.146336,0.288092,0.367038,10.275659,4.777425,0.109772,1.427095,0.069986
737,en_full_macro,XLF,2009-03-31,2009-04-30,0.062968,-1.343476,0.751981,0.380037,6.035425,2.021106,0.100533,0.963335,0.087363
465,en_vix_only,XLF,2008-10-31,2008-11-30,0.087358,-1.210195,0.550365,0.392581,4.493921,1.506862,0.093161,0.725249,0.059396
519,en_no_vix,XLF,2008-11-30,2008-12-31,0.076171,-1.407779,0.761574,0.358082,4.701030,1.361876,0.079474,0.760501,0.165705
